# Problem 4 — Site Allow/Deny-List Agent (CrowdStrike Falcon)

> *GIS requests a site to be whitelisted or blacklisted. Automate the end-to-end
> fulfillment of this operation.*

**Focused scope:** this notebook fulfills the request **entirely through CrowdStrike Falcon
Custom IOC management** — the endpoint-agent enforcement layer. A *blacklist* is a Falcon
IOC with `action="prevent"`; a *whitelist* is an IOC with `action="allow"`.

## One thing to know up front about "URL filtering" in CrowdStrike
CrowdStrike custom IOCs are **domain / IPv4 / IPv6** indicators — there is **no full-URL
(path) indicator type**. So "filter `https://bad.example/login.php`" becomes a **domain**
IOC for `bad.example` (which matches that host and its subdomains). Full path-level URL
filtering would need a web proxy/SWG; at the Falcon layer we operate on the domain/IP. The
agent does this reduction automatically and says so.

## What this agent does
A small **LangGraph** workflow:

1. **Intake** — read the ServiceNow request (indicator, allow/deny, scope, duration).
2. **Triage** — reduce the indicator to a Falcon type (`domain`/`ipv4`/`ipv6`), pull
   threat-intel reputation, and apply hard guardrails.
3. **Plan** — map the request to Falcon IOC parameters (`action`, `severity`, `platforms`,
   `applied_globally`/`host_groups`, `expiration`).
4. **Approve** — pause for a human (always for whitelist; always when reputation is bad).
5. **Apply** — `indicator_create` via the Falcon IOC API, then `indicator_search`+`get` to
   **verify** enforcement; close the ticket.

```
 SNOW request ─► triage ─► plan ─► [approve] ─► apply (Falcon indicator_create)
  (allow/deny)   (reduce   (IOC    interrupt()      │
                  +guard)  params)              verify (search+get) ─► close
```

**Guardrails that override the model**
- **Never** create a `prevent` IOC for critical infrastructure (`microsoft.com`, Windows
  Update, identity providers, your own domains) — refused and escalated.
- **Never** auto-`allow` a known-malicious indicator — escalated for senior approval.
- Whitelist (`allow`) entries are time-boxed (`expiration`) and always human-approved.

Everything runs offline against a **mock that mirrors the FalconPy `IOC` interface**;
swapping to production is one line (`FALCON = IOC(...)`). The real API + CLI usage is in §8.

## 0. Setup
Runs with **no credentials** — ServiceNow, threat-intel, and CrowdStrike are simulated.
Set an LLM key in `.env` for real planning; otherwise a deterministic stub runs the flow.

In [ ]:
import os, re, json, datetime as dt
from typing import TypedDict, Literal, Optional
from urllib.parse import urlparse
from pydantic import BaseModel, Field
import pandas as pd

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command
from langchain_core.messages import SystemMessage, HumanMessage

try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
print("Imports OK")

In [ ]:
def get_secret(name, default=None):
    v = os.getenv(name)
    if v:
        return v
    try:
        with open("config.local.json") as f:
            return json.load(f).get(name, default)
    except FileNotFoundError:
        return default

AUDIT = []
def audit(action, target, detail=None, actor="crowdstrike-agent"):
    rec = {"ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds"),
           "actor": actor, "action": action, "target": target, "detail": detail or {}}
    AUDIT.append(rec)
    print(f"  [audit] {action:26s} {target}  {detail or ''}")
    return rec

def build_llm(temperature=0, model=None):
    provider = os.getenv("LLM_PROVIDER", "anthropic").lower()
    model = model or os.getenv("LLM_MODEL")
    try:
        if provider == "anthropic":
            key = get_secret("ANTHROPIC_API_KEY")
            if not key:
                return None
            from langchain_anthropic import ChatAnthropic
            return ChatAnthropic(model=model or "claude-sonnet-4-6",
                                 temperature=temperature, api_key=key, max_tokens=2048)
        if provider == "azure_openai":
            key = get_secret("AZURE_OPENAI_API_KEY")
            if not key:
                return None
            from langchain_openai import AzureChatOpenAI
            return AzureChatOpenAI(
                azure_deployment=os.getenv("AZURE_OPENAI_DEPLOYMENT", "gpt-4.1"),
                api_version=os.getenv("AZURE_OPENAI_API_VERSION", "2024-10-21"),
                azure_endpoint=get_secret("AZURE_OPENAI_ENDPOINT"),
                api_key=key, temperature=temperature)
        if provider == "openai":
            key = get_secret("OPENAI_API_KEY")
            if not key:
                return None
            from langchain_openai import ChatOpenAI
            return ChatOpenAI(model=model or "gpt-4.1", api_key=key, temperature=temperature)
    except Exception as e:
        print("build_llm: offline stub ->", e)
        return None
    return None

LLM = build_llm()
LLM_AVAILABLE = LLM is not None
print("LLM:", "LIVE (" + os.getenv("LLM_PROVIDER", "anthropic") + ")" if LLM_AVAILABLE
      else "OFFLINE deterministic stub (set an API key in .env to enable real reasoning)")

def llm_structured(system, user, schema, demo):
    if LLM is None:
        return demo
    return LLM.with_structured_output(schema).invoke(
        [SystemMessage(content=system), HumanMessage(content=user)])

## 1. CrowdStrike Falcon IOC client (mock that mirrors FalconPy)

The mock implements the same surface as `falconpy.IOC` — `indicator_create`,
`indicator_search`, `indicator_get`, `indicator_delete` — and returns the same
`{status_code, body:{resources, errors}}` shape. To go live, delete the mock and
uncomment the `IOC(...)` line: **the agent code below does not change**.

In [ ]:
class MockFalconIOC:
    """In-memory stand-in for falconpy.IOC. Same method names + response envelope."""
    def __init__(self):
        self._store = {}      # id -> indicator dict
        self._seq = 0

    def indicator_create(self, **f):
        # FalconPy accepts these as kwargs (Body Payload abstraction) or body={"indicators":[...]}.
        self._seq += 1
        ioc_id = f"ioc_{self._seq:05d}"
        rec = {"id": ioc_id, "type": f.get("type"), "value": f.get("value"),
               "action": f.get("action"), "severity": f.get("severity"),
               "platforms": f.get("platforms"), "applied_globally": f.get("applied_globally"),
               "host_groups": f.get("host_groups"), "expiration": f.get("expiration"),
               "description": f.get("description"), "source": f.get("source"),
               "tags": f.get("tags")}
        self._store[ioc_id] = rec
        audit("falcon.indicator_create", rec["value"],
              {"id": ioc_id, "type": rec["type"], "action": rec["action"]})
        return {"status_code": 201, "body": {"resources": [rec], "errors": []}}

    def indicator_search(self, filter=None, **kw):
        """Minimal FQL support: value:'x' and/or type:'y'."""
        want = dict(re.findall(r"(\w+):'([^']*)'", filter or ""))
        ids = [i for i, r in self._store.items()
               if all(str(r.get(k)) == v for k, v in want.items())]
        return {"status_code": 200, "body": {"resources": ids, "errors": []}}

    def indicator_get(self, ids=None, **kw):
        ids = ids or []
        if isinstance(ids, str):
            ids = [ids]
        return {"status_code": 200,
                "body": {"resources": [self._store[i] for i in ids if i in self._store]}}

    def indicator_delete(self, ids=None, **kw):
        ids = [ids] if isinstance(ids, str) else (ids or [])
        for i in ids:
            self._store.pop(i, None)
        audit("falcon.indicator_delete", ",".join(ids))
        return {"status_code": 200, "body": {"resources": ids, "errors": []}}

FALCON = MockFalconIOC()

# === REAL API (drop-in) ===
# pip install crowdstrike-falconpy
# from falconpy import IOC
# FALCON = IOC(client_id=get_secret("FALCON_CLIENT_ID"),
#              client_secret=get_secret("FALCON_CLIENT_SECRET"),
#              base_url=os.getenv("FALCON_BASE_URL", "us-1"))   # us-1|us-2|eu-1|us-gov-1
# Required API scope: "IOC Management" (Read & Write).
print("Falcon IOC client ready:", type(FALCON).__name__)

## 2. ServiceNow intake, threat-intel, and guardrails

In [ ]:
# Requests from GIS. `https://bad.example/login.php` shows the URL -> domain reduction.
REQUESTS = [
    {"number": "REQ0101", "indicator": "malware-c2.example",          "action": "deny",
     "scope": "enterprise", "duration_days": None, "requested_by": "soc.analyst",
     "justification": "Confirmed C2 from incident INC0456"},
    {"number": "REQ0102", "indicator": "vendor-portal.partner.com",   "action": "allow",
     "scope": "Finance-Workstations", "duration_days": 90, "requested_by": "ap.manager",
     "justification": "New AP vendor portal flagged by detection"},
    {"number": "REQ0103", "indicator": "microsoft.com",               "action": "deny",
     "scope": "enterprise", "duration_days": None, "requested_by": "helpdesk.t1",
     "justification": "User thinks a 'microsoft popup' is malware"},
    {"number": "REQ0104", "indicator": "free-prizes-login.ru",        "action": "allow",
     "scope": "enterprise", "duration_days": 30, "requested_by": "marketing.intern",
     "justification": "Campaign landing page"},
    {"number": "REQ0105", "indicator": "https://bad.example/login.php", "action": "deny",
     "scope": "enterprise", "duration_days": None, "requested_by": "soc.analyst",
     "justification": "Phishing kit URL from user report"},
    {"number": "REQ0106", "indicator": "203.0.113.66",               "action": "deny",
     "scope": "enterprise", "duration_days": None, "requested_by": "soc.analyst",
     "justification": "Scanning our perimeter"},
]

def snow_fetch_requests():
    return list(REQUESTS)
    # === REAL API === GET {SNOW}/api/now/table/sc_req_item?sysparm_query=cat_item=allowdeny^active=true

def snow_update(number, state, work_notes=""):
    audit("servicenow.update", number, {"state": state, "notes": work_notes[:70]})
    return {"number": number, "state": state}
    # === REAL API === PATCH {SNOW}/api/now/table/sc_req_item/{sys_id}

_REPUTATION = {
    "malware-c2.example": {"verdict": "malicious", "score": 92},
    "free-prizes-login.ru": {"verdict": "malicious", "score": 88},
    "vendor-portal.partner.com": {"verdict": "clean", "score": 3},
    "bad.example": {"verdict": "malicious", "score": 90},
    "203.0.113.66": {"verdict": "suspicious", "score": 61},
    "microsoft.com": {"verdict": "clean", "score": 0},
}
def threat_intel(value):
    return _REPUTATION.get(value, {"verdict": "unknown", "score": 50})
    # === REAL API === VirusTotal / CrowdStrike Falcon Intelligence / internal TIP

CRITICAL_NEVER_BLOCK = {
    "microsoft.com", "windowsupdate.com", "office.com", "office365.com",
    "login.microsoftonline.com", "azure.com", "contoso.com", "okta.com",
    "google.com", "apple.com", "amazonaws.com", "crowdstrike.com",
}

In [ ]:
IPV4 = re.compile(r"^\d{1,3}(\.\d{1,3}){3}$")

def to_falcon_indicator(raw):
    """Reduce a request indicator to a CrowdStrike (type, value). URLs -> domain host."""
    raw = raw.strip()
    if IPV4.match(raw):
        return "ipv4", raw
    if ":" in raw and raw.count(":") >= 2 and "/" not in raw:
        return "ipv6", raw
    if "://" in raw or "/" in raw:
        host = urlparse(raw if "://" in raw else "http://" + raw).hostname
        return "domain", (host or raw.split("/")[0])
    return "domain", raw

def registrable(domain):
    parts = domain.split(".")
    return ".".join(parts[-2:]) if len(parts) >= 2 else domain

def guardrail_verdict(action, cs_type, value, rep):
    """PROCEED / REFUSE / ESCALATE — code-enforced, overrides the LLM."""
    if action == "deny" and cs_type == "domain":
        if value in CRITICAL_NEVER_BLOCK or registrable(value) in CRITICAL_NEVER_BLOCK:
            return "REFUSE", f"{value} is critical infrastructure — blocking is prohibited"
    if action == "allow" and rep["verdict"] == "malicious":
        return "ESCALATE", f"cannot auto-allow a known-malicious indicator (rep={rep['score']})"
    return "PROCEED", "passed guardrails"

## 3. Agent state + IOC plan schema

In [ ]:
class IOCPlan(BaseModel):
    request_id: str
    indicator_value: str
    cs_type: Literal["domain", "ipv4", "ipv6"]
    cs_action: Literal["prevent", "detect", "allow"]   # prevent=blacklist, allow=whitelist
    severity: Literal["informational", "low", "medium", "high", "critical"]
    platforms: list[Literal["windows", "mac", "linux"]]
    applied_globally: bool
    host_groups: list[str]
    expiration: Optional[str] = None
    verdict: Literal["PROCEED", "REFUSE", "ESCALATE"]
    rationale: str

class AgentState(TypedDict):
    requests: list[dict]
    triaged: list[dict]
    plans: list[dict]
    approved: list[str]
    results: list[dict]
    report: dict

In [ ]:
TODAY = dt.date(2026, 6, 2)

def rule_plan(t) -> IOCPlan:
    """Deterministic offline planner — mirrors CrowdStrike best practice + guardrails."""
    deny = t["action"] == "deny"
    exp = None
    if t["duration_days"]:
        exp = (TODAY + dt.timedelta(days=t["duration_days"])).isoformat() + "T00:00:00Z"
    glob = t["scope"] == "enterprise"
    return IOCPlan(
        request_id=t["number"], indicator_value=t["cs_value"], cs_type=t["cs_type"],
        cs_action="prevent" if deny else "allow",
        severity="high" if deny else "informational",
        platforms=["windows", "mac", "linux"],
        applied_globally=glob, host_groups=[] if glob else [t["scope"]],
        expiration=exp, verdict=t["guardrail"], rationale=t["guardrail_reason"])

## 4. Graph nodes

In [ ]:
def n_intake(state: AgentState) -> dict:
    reqs = snow_fetch_requests()
    audit("intake.requests", "servicenow", {"count": len(reqs)})
    return {"requests": reqs}

def n_triage(state: AgentState) -> dict:
    triaged = []
    for req in state["requests"]:
        cs_type, cs_value = to_falcon_indicator(req["indicator"])
        rep = threat_intel(cs_value)
        verdict, reason = guardrail_verdict(req["action"], cs_type, cs_value, rep)
        reduced = " (reduced from URL)" if cs_value != req["indicator"] else ""
        triaged.append({**req, "cs_type": cs_type, "cs_value": cs_value,
                        "reputation": rep, "guardrail": verdict, "guardrail_reason": reason})
        print(f"[triage] {req['number']} {req['action']:5s} {req['indicator']:30s} "
              f"-> {cs_type}:{cs_value}{reduced} rep={rep['verdict']:9s} {verdict}")
    return {"triaged": triaged}

In [ ]:
PLAN_SYS = (
    "You configure CrowdStrike Falcon custom IOCs. Map an allow/deny request to IOC "
    "parameters. deny -> action 'prevent' (severity high); allow -> action 'allow' "
    "(severity informational). platforms = [windows, mac, linux] unless told otherwise. "
    "scope 'enterprise' -> applied_globally true with empty host_groups; any other scope "
    "-> applied_globally false with that scope as a host group. If duration_days is set, "
    "set an ISO8601 UTC expiration. Respect the guardrail verdict: if REFUSE or ESCALATE, "
    "return that verdict and do not enforce. One-line rationale."
)

def n_plan(state: AgentState) -> dict:
    plans = []
    for t in state["triaged"]:
        demo = rule_plan(t)
        user = json.dumps({k: t[k] for k in ("number", "indicator", "cs_type", "cs_value",
                          "action", "scope", "duration_days", "reputation",
                          "guardrail", "guardrail_reason")}, default=str)
        plan = llm_structured(PLAN_SYS, "Plan this Falcon IOC:\n" + user, IOCPlan, demo)
        if t["guardrail"] != "PROCEED":          # guardrail always wins
            plan.verdict = t["guardrail"]
        p = plan.model_dump(); p["_req"] = t
        plans.append(p)
        tag = (f"{p['cs_action']} sev={p['severity']} "
               f"{'global' if p['applied_globally'] else p['host_groups']} "
               f"exp={p['expiration'] or 'never'}") if p["verdict"] == "PROCEED" else "-"
        print(f"[plan]   {p['request_id']} {p['verdict']:8s} {tag}")
    return {"plans": plans}

In [ ]:
def n_approval(state: AgentState) -> dict:
    actionable = [p for p in state["plans"] if p["verdict"] == "PROCEED"]
    for p in state["plans"]:                      # route non-PROCEED to humans now
        if p["verdict"] == "REFUSE":
            snow_update(p["request_id"], "closed_rejected", p["rationale"])
        elif p["verdict"] == "ESCALATE":
            snow_update(p["request_id"], "escalated", p["rationale"])
    if not actionable:
        return {"approved": []}
    decision = interrupt({
        "type": "approve_falcon_iocs",
        "summary": f"{len(actionable)} Falcon IOC change(s) ready",
        "changes": [{"request": p["request_id"], "action": p["cs_action"],
                     "type": p["cs_type"], "value": p["indicator_value"],
                     "scope": "global" if p["applied_globally"] else p["host_groups"],
                     "reputation": p["_req"]["reputation"]["verdict"]} for p in actionable],
        "note": "action='allow' weakens detection — confirm whitelist entries.",
        "instructions": "resume with {'approved': [request ids]} or {'approved': 'ALL'}",
    })
    approved = decision.get("approved", []) if isinstance(decision, dict) else []
    if approved == "ALL":
        approved = [p["request_id"] for p in actionable]
    audit("human.approval", "falcon-ioc-batch", {"approved": approved})
    return {"approved": approved}

In [ ]:
def n_apply(state: AgentState) -> dict:
    by_id = {p["request_id"]: p for p in state["plans"]}
    results = []
    for rid in state["approved"]:
        p = by_id[rid]
        # Build the indicator; CrowdStrike wants applied_globally XOR host_groups.
        fields = {"type": p["cs_type"], "value": p["indicator_value"],
                  "action": p["cs_action"], "severity": p["severity"],
                  "platforms": p["platforms"], "expiration": p["expiration"],
                  "description": f"{rid}: {p['_req']['justification']}",
                  "source": f"ServiceNow {rid}", "tags": ["gis-automation", p["cs_action"]]}
        if p["applied_globally"]:
            fields["applied_globally"] = True
        else:
            fields["host_groups"] = p["host_groups"]
        fields = {k: v for k, v in fields.items() if v not in (None, [])}

        resp = FALCON.indicator_create(**fields)
        created = resp["status_code"] in (200, 201) and resp["body"]["resources"]

        # Verify: search by value+type, then get, and confirm the action stuck.
        sr = FALCON.indicator_search(filter=f"value:'{p['indicator_value']}'+type:'{p['cs_type']}'")
        got = FALCON.indicator_get(ids=sr["body"]["resources"])["body"]["resources"]
        verified = any(g["action"] == p["cs_action"] for g in got)

        ok = bool(created) and verified
        snow_update(rid, "closed_complete" if ok else "work_in_progress",
                    f"Falcon IOC {p['cs_action']} on {p['cs_type']} {p['indicator_value']}"
                    if ok else "create/verify failed")
        results.append({"request_id": rid, "type": p["cs_type"],
                        "value": p["indicator_value"], "action": p["cs_action"],
                        "ioc_id": resp["body"]["resources"][0]["id"] if created else None,
                        "verified": verified, "ok": ok})
        print(f"[apply]  {rid} {p['cs_action']:7s} {p['cs_type']}:{p['indicator_value']:24s} "
              f"id={results[-1]['ioc_id']} verified={verified}")
    return {"results": results}

def n_report(state: AgentState) -> dict:
    enforced = [r for r in state["results"] if r["ok"]]
    report = {"requests": len(state["requests"]),
              "enforced": [r["request_id"] for r in enforced],
              "blacklisted": [r["value"] for r in enforced if r["action"] == "prevent"],
              "whitelisted": [r["value"] for r in enforced if r["action"] == "allow"],
              "refused": [p["request_id"] for p in state["plans"] if p["verdict"] == "REFUSE"],
              "escalated": [p["request_id"] for p in state["plans"] if p["verdict"] == "ESCALATE"]}
    print(f"[report] {report}")
    return {"report": report}

## 5. Build & compile the graph

In [ ]:
g = StateGraph(AgentState)
for name, fn in [("intake", n_intake), ("triage", n_triage), ("plan", n_plan),
                 ("approval", n_approval), ("apply", n_apply), ("report", n_report)]:
    g.add_node(name, fn)
g.add_edge(START, "intake")
g.add_edge("intake", "triage")
g.add_edge("triage", "plan")
g.add_edge("plan", "approval")
g.add_edge("approval", "apply")
g.add_edge("apply", "report")
g.add_edge("report", END)
app = g.compile(checkpointer=MemorySaver())
print("Graph compiled:", list(app.get_graph().nodes))

## 6. Run it

In [ ]:
AUDIT.clear()
config = {"configurable": {"thread_id": "falcon-run-2026-06-02"}}
init = {"requests": [], "triaged": [], "plans": [], "approved": [], "results": [], "report": {}}
result = app.invoke(init, config)

print("\n=== PAUSED FOR APPROVAL ===")
intr = result["__interrupt__"][0].value
print(intr["summary"], "--", intr["note"])
for c in intr["changes"]:
    print(f"  - {c['request']}  {c['action']:7s} {c['type']}:{c['value']:24s} "
          f"scope={c['scope']}  rep={c['reputation']}")

In [ ]:
final = app.invoke(Command(resume={"approved": "ALL"}), config)
print("\n=== FINAL REPORT ===")
print(json.dumps(final["report"], indent=2))

## 7. Results + audit trail

In [ ]:
rows = [{"request": p["request_id"], "action": p["cs_action"], "type": p["cs_type"],
         "value": p["indicator_value"], "verdict": p["verdict"],
         "scope": "global" if p["applied_globally"] else p["host_groups"],
         "expiration": p["expiration"], "rationale": p["rationale"]} for p in final["plans"]]
display(pd.DataFrame(rows))
print()
display(pd.DataFrame(AUDIT)[["action", "target", "detail"]])

## 8. Using the real CrowdStrike API / CLI

The mock above mirrors FalconPy exactly, so going live is a one-line swap. Three ways to
drive Falcon Custom IOCs:

### A. Python — FalconPy SDK (recommended; what this notebook is written against)
```python
pip install crowdstrike-falconpy
from falconpy import IOC
falcon = IOC(client_id="...", client_secret="...", base_url="us-1")  # us-1|us-2|eu-1|us-gov-1

# Blacklist a domain (block on the endpoint), all platforms, enterprise-wide:
falcon.indicator_create(type="domain", value="malware-c2.example", action="prevent",
                        severity="high", platforms=["windows", "mac", "linux"],
                        applied_globally=True, source="ServiceNow REQ0101",
                        description="Confirmed C2 from INC0456", tags=["gis-automation"])

# Whitelist a domain for one host group, expiring in 90 days:
falcon.indicator_create(type="domain", value="vendor-portal.partner.com", action="allow",
                        severity="informational", platforms=["windows", "mac", "linux"],
                        host_groups=["<group_id>"], expiration="2026-08-31T00:00:00Z")

# Verify / remove:
ids = falcon.indicator_search(filter="value:'malware-c2.example'+type:'domain'")["body"]["resources"]
falcon.indicator_get(ids=ids)
falcon.indicator_delete(ids=ids)
```
Required API client scope: **IOC Management — Read & Write**.

### B. CLI — `curl` against the Falcon REST API
```bash
# 1) OAuth2 token
TOKEN=$(curl -s -X POST https://api.crowdstrike.com/oauth2/token \
  -d "client_id=$FALCON_CLIENT_ID&client_secret=$FALCON_CLIENT_SECRET" | jq -r .access_token)

# 2) Create a blacklist (prevent) domain IOC
curl -s -X POST https://api.crowdstrike.com/iocs/entities/indicators/v1 \
  -H "Authorization: Bearer $TOKEN" -H "Content-Type: application/json" \
  -d '{"indicators":[{"type":"domain","value":"malware-c2.example","action":"prevent",
       "severity":"high","platforms":["windows","mac","linux"],"applied_globally":true,
       "source":"ServiceNow REQ0101","description":"C2 from INC0456"}]}'
```

### C. CLI — PowerShell (`PSFalcon`)
```powershell
Install-Module -Name PSFalcon -Scope CurrentUser
Request-FalconToken -ClientId '...' -ClientSecret '...' -Cloud us-1
New-FalconIoc -Type domain -Value 'malware-c2.example' -Action prevent -Severity high `
              -Platforms windows,mac,linux -AppliedGlobally $true -Source 'ServiceNow REQ0101'
```

## 9. Productionizing
- **Indicator scope:** Falcon IOCs are **domain / IP** — a URL request is enforced at the
  domain level (the agent reduces it). For path-level URL filtering, pair this with an SWG;
  Falcon covers the endpoint-agent layer of the original problem statement.
- **`applied_globally` vs `host_groups`:** they're mutually exclusive. Map the request
  *scope* to host group IDs (look them up via the Host Groups API) for targeted changes.
- **Whitelists expire:** always set `expiration` on `allow` entries; run a reconcile job to
  catch any that slipped through without one (the top audit finding).
- **Normalize indicators:** apply eTLD+1 + IDN/punycode handling before the never-block
  check so look-alikes can't dodge it; dedupe against existing IOCs (`indicator_search`).
- **Least privilege + idempotency:** a dedicated API client scoped to IOC Management;
  key IOCs by request id (`source`) so re-runs and rollbacks are clean.
- **Close the loop:** Falcon raises detections when a blocked IOC is hit — feed those back
  to the SOC and to the requesting ticket as enforcement evidence.